# Walkthroughs and Exercises for Deep Learning for Business Made Simple

**Dr. Chester Ismay**

We teach in **Keras 3** running on the **PyTorch** backend (the same code also runs
on **TensorFlow** -- see the touchpoints throughout). Run the setup cell first, then
work top to bottom. All data is fetched from the web (a GitHub URL for the hotel
data; Hugging Face streaming for the image and text data), so there is **nothing to
upload** -- each dataset loads itself when you run its cell.


In [ ]:
# Run this once per session, then RESTART the runtime if Colab prompts you to.
# Colab already ships torch; we add/upgrade Keras 3 and the datasets library.
!pip install -q -U keras datasets
!pip install -q scikit-learn matplotlib pillow


In [ ]:
# Run this setup cell FIRST. The backend must be chosen BEFORE keras is imported.
import os
os.environ["KERAS_BACKEND"] = "torch"

import keras
import numpy as np
import pandas as pd

# Display all columns and all outputs from each cell
pd.set_option("display.max_columns", None)
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

keras.utils.set_random_seed(2026)
print("Keras", keras.__version__, "on backend:", keras.backend.backend())

# Intro: Getting Started with Deep Learning for Business

In this course we use **Keras 3**, a friendly, high-level deep learning API,
running on the **PyTorch** backend. You write simple Keras code; PyTorch does the
heavy lifting underneath. The solutions file also includes an **idiomatic PyTorch**
version of every walkthrough so you can see what the same model looks like in raw
PyTorch.

## One Keras, three engines (TensorFlow, PyTorch, or JAX)

Keras 3 is **engine-agnostic**: the *same* model code runs on **TensorFlow**, **PyTorch**,
or **JAX**. You choose the engine once per session by setting `KERAS_BACKEND` **before**
importing Keras; nothing else in this course changes. We use PyTorch here, to run any of
it on TensorFlow instead, you'd change one line in the setup cell and restart the runtime:

<div align="center"><img src="https://raw.githubusercontent.com/ismayc/oreilly-deep-learning-made-simple/main/assets/diagrams/diagram-080b8df74363.png" alt="Mermaid diagram" width="654"></div>

A quick aside on the three engines, since you will hear all of them mentioned. Here is
the short version of when each one shines:

| Engine | Strengths | Trade-offs |
|---|---|---|
| **PyTorch** *(our choice)* | Pythonic and easy to debug; dominant in research and now common in industry; huge ecosystem of models and tutorials | Production serving tooling was historically less turnkey than TensorFlow's |
| **TensorFlow** | Mature deployment stack (TF Serving, TFLite, on-device); battle-tested at scale | More boilerplate; less convenient for quick, interactive experimentation |
| **JAX** | Very fast, especially on TPUs; composable `jit`, `grad`, and `vmap` for cutting-edge research | Smaller ecosystem and a steeper learning curve; least beginner-friendly |

We use **PyTorch** because it is the friendliest to learn and debug and the backend you
are most likely to meet on the job. Because Keras 3 is engine-agnostic, it is a low-stakes
choice either way.

In [ ]:
# To run this course on the TensorFlow engine instead of PyTorch, set this BEFORE
# `import keras`, then restart the runtime (the engine is locked in once Keras imports).
import os
os.environ["KERAS_BACKEND"] = "tensorflow"   # was "torch"

In [ ]:
# Whichever engine is active, this is how you check it:
print("Active Keras engine:", keras.backend.backend())

Keras can also target: `tensorflow`, `torch`, and `jax`.

> **Why this matters for you:** you learn *one* high-level API and stay portable. If your
> team standardizes on TensorFlow (or JAX) later, your Keras code comes along, only the
> one-line engine setting changes.

## Walkthrough: Setting Up Keras (PyTorch backend) and Your First Tensor

### Confirm the environment

In [ ]:
import keras, torch
print("Keras backend:", keras.backend.backend())   # should say 'torch'
print("PyTorch version:", torch.__version__)

> **Common Pitfall:** `os.environ["KERAS_BACKEND"] = "torch"` must run **before**
> `import keras`. If you import keras first and then set it, you'll silently get
> the default backend. When in doubt, restart the kernel and run the setup cell first.

### A first tensor operation

In [ ]:
# A tensor is just an array the framework can run fast math on (and learn from)
x = keras.ops.array([[1.0, 2.0], [3.0, 4.0]])
print("Sum of all elements:", float(keras.ops.sum(x)))
print("Column means:", keras.ops.mean(x, axis=0))

## Exercise: Confirm Your Environment

In [ ]:
# Build a tiny one-line "model" and run data through it -- proves the stack works
layer = keras.layers.Dense(1, activation="sigmoid")
out = layer(keras.ops.array([[0.5, -1.2, 3.0]]))
print("Output shape:", out.shape)

If you see a `(1, 1)` tensor above, you are ready to go.

> **GenAI tip:** If a setup cell errors, copy the *full* error plus one line of context
> ("I'm using Keras 3 with the PyTorch backend on Colab") into your AI assistant and
> ask for the fix. Re-run to confirm, never trust a fix you haven't re-run.

# Module 1: Demystifying Neural Networks for Business Insight

We'll predict **hotel booking cancellations**, a real revenue-management problem.
Every cancellation is a room you could have sold. The dataset has ~119,000 real
bookings.

## Walkthrough 1.1: Predicting Booking Cancellations with a Neural Network

### Load and inspect the data

In [ ]:
url = ("https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/"
       "data/2020/2020-02-11/hotels.csv")
hotels = pd.read_csv(url)
print(hotels.shape)
hotels[["hotel", "lead_time", "adults", "previous_cancellations",
        "adr", "is_canceled"]].head()

<details>
<summary><strong>Data dictionary</strong> (click to expand) — the hotel-booking columns we use</summary>

| Column | Type | What it stores |
|---|---|---|
| `hotel` | text | Property type: "City Hotel" or "Resort Hotel" |
| `lead_time` | integer | Days between the booking and the arrival date |
| `stays_in_weekend_nights` | integer | Weekend nights (Sat/Sun) in the stay |
| `stays_in_week_nights` | integer | Weeknights (Mon to Fri) in the stay |
| `adults` | integer | Number of adults on the booking |
| `previous_cancellations` | integer | Earlier bookings this guest had canceled |
| `booking_changes` | integer | Changes made to the booking after it was placed |
| `total_of_special_requests` | integer | Count of special requests (e.g., a high floor) |
| `adr` | float | Average daily rate: the average price per night |
| `is_canceled` | binary (0/1) | Target: 1 if the booking was canceled, 0 if kept |

</details>

In [ ]:
# How often do guests cancel? (our target)

### Prepare the features

<div align="center"><img src="https://raw.githubusercontent.com/ismayc/oreilly-deep-learning-made-simple/main/assets/diagrams/diagram-d05a14f41739.png" alt="Mermaid diagram" width="481"></div>

In [ ]:
# A handful of numeric drivers a revenue manager would recognize




# Neural networks train far better on standardized inputs. Fit the shift/scale on the
# TRAINING data only, then apply the SAME numbers to validation (see the pitfall below).

> **Common Pitfall:** Standardize using statistics from the **training** data only,
> then apply the same shift/scale to validation and new data. Computing the mean on
> all the data leaks information about the future into training.

### Build, train, and evaluate the network

<div align="center"><img src="https://raw.githubusercontent.com/ismayc/oreilly-deep-learning-made-simple/main/assets/diagrams/diagram-e6adc6c67bed.png" alt="Mermaid diagram" width="872"></div>

<div align="center"><img src="https://raw.githubusercontent.com/ismayc/oreilly-deep-learning-made-simple/main/assets/diagrams/diagram-fb5e4882d46b.png" alt="Mermaid diagram" width="679"></div>

In [ ]:
import matplotlib.pyplot as plt
plt.plot(history.history["loss"], label="train loss")
plt.plot(history.history["val_loss"], label="validation loss")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend()
plt.title("The falling loss IS the model learning")
plt.show()

### Visualize: can the model tell the two groups apart?

In [ ]:
# A model that "separates" the classes pushes cancellations toward 1 and the rest
# toward 0. Overlapping histograms = an indecisive model.
val_probs = model.predict(X_val, verbose=0).ravel()
plt.hist(val_probs[y_val == 0], bins=30, alpha=0.6, label="actually kept")
plt.hist(val_probs[y_val == 1], bins=30, alpha=0.6, label="actually canceled")
plt.axvline(0.5, color="k", ls="--", lw=1, label="default threshold")
plt.xlabel("predicted cancellation probability"); plt.ylabel("number of bookings")
plt.legend(); plt.title("Separation is the model's confidence, made visible")
plt.show()

> **GenAI tip:** Wondering whether ~78% accuracy is actually good? Paste the class
> balance and the validation accuracy into an AI assistant and ask, "is this better than
> always guessing the majority class, and by how much?". Then check it against the
> baseline we compute in Module 4.

## Exercise 1.1: Predicting Average Daily Rate (Regression)

Now the target is a **number**, the average daily rate (`adr`), i.e. price. The
same network shape works; you just change the last layer and the loss.

<div align="center"><img src="https://raw.githubusercontent.com/ismayc/oreilly-deep-learning-made-simple/main/assets/diagrams/diagram-ab2f8d9b476b.png" alt="Mermaid diagram" width="606"></div>

In [ ]:
# Data prep is the same recipe as the walkthrough -- run it and move on.
reg_features = hotels[features].copy()
adr_price = hotels["adr"].astype("float32")
# Keep rows with no missing features, a valid price, and a sensible range (drop outliers > $1000)
keep = reg_features.notna().all(axis=1) & adr_price.notna() & (hotels["adr"].between(0, 1000))
X_reg = reg_features[keep].drop(columns=["adr"]).values.astype("float32")
y_reg = adr_price[keep].values
X_reg_train, X_reg_val, y_reg_train, y_reg_val = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=2026)
# Standardize on the training split only, then apply to validation (same rule as the walkthrough)
reg_mean, reg_std = X_reg_train.mean(0), X_reg_train.std(0) + 1e-8
X_reg_train = (X_reg_train - reg_mean) / reg_std
X_reg_val = (X_reg_val - reg_mean) / reg_std
print("Regression features:", X_reg.shape)

In [ ]:
# We'll build this together. Two choices flip this from a classifier to a REGRESSION
# model -- notice each one as we go:
#   * the output Dense layer has NO activation, so it can predict ANY dollar amount
#   * the loss is "mse" (mean squared error), which measures how many dollars off we are

In [ ]:
# Visualize results: predicted vs actual price (points on the dashed line = perfect)
pred_adr = reg_model.predict(X_reg_val, verbose=0).ravel()
plt.scatter(y_reg_val, pred_adr, s=6, alpha=0.25)
lim = [0, 400]
plt.plot(lim, lim, "r--", label="perfect prediction")
plt.xlim(lim); plt.ylim(lim)
plt.xlabel("actual ADR ($)"); plt.ylabel("predicted ADR ($)")
plt.legend(); plt.title("Where the price model is tight -- and where it drifts")
plt.show()

> **Common Pitfall:** For regression, the output layer has **no activation** and the
> loss is `mse` (or `mae`), not `sigmoid` + `binary_crossentropy`. Using a sigmoid
> here would squash every prediction between 0 and 1.

## Walkthrough + Exercise 1.2: Use GenAI to Decode Errors and Explain Choices

Models throw confusing errors while you build them, and it isn't always obvious why a
layer is shaped the way it is. Your AI assistant is a fast debugging and explanation
partner, the skill is prompting clearly and **verifying** (never trust a fix you
haven't re-run).

**A prompt to start from** (paste into your AI assistant, then iterate):

> I'm using Keras 3 on the PyTorch backend. Here is my code and the full error message:
> [paste]. Explain the cause in plain English, give the smallest fix, then explain in one
> sentence each what my hidden Dense layers and the sigmoid output layer are doing.

In [ ]:
# Scratch space for 1.2 -- paste an error or a model summary here and work with your
# AI assistant. (Nothing to run yet; this cell is yours to experiment in.)

> **GenAI tip:** Apply the suggested fix, then **re-run the cell** to confirm it works.
> If the assistant invents a function or argument, check it against the Keras docs.

### Interpretation Questions

1. The classification model reaches ~78-80% accuracy. With ~63% of bookings *not*
   canceled, is that a strong result? (Hint: what would "always predict not-canceled"
   score? We dig into this in Module 4.)
2. The regression model's MAE is in **dollars**. Is being off by that much acceptable
   for a pricing decision? Where would it need to be tighter?
3. Which features would you add to improve either model, and which might be
   *leakage* (information you wouldn't have at booking time)?

### Self-Check

By the end of this module, you should be able to:

- [ ] Load a business dataset and standardize numeric features for a network
- [ ] Build, compile, and train a feedforward network in Keras
- [ ] Switch a model between **classification** (sigmoid + cross-entropy) and **regression** (no activation + MSE)
- [ ] Read a falling loss curve as evidence of learning
- [ ] Recognize the same model written in idiomatic PyTorch (see the solutions file)

# Module 2: Deep Learning for Images and Customer Experience

Direct application: **auto-tagging product/food photos**, the kind of task behind
delivery-app menus, retail catalogs, and visual search. We use the **Food-101**
dataset and a **pre-trained** convolutional network, so we get strong results without
training on millions of images ourselves (transfer learning).

## Walkthrough 2.1: Auto-Tagging Photos with a Pre-Trained CNN

### Grab a few images

In [ ]:
from datasets import load_dataset
from PIL import Image

food = load_dataset("ethz/food101", split="train", streaming=True)
class_names = food.features["label"].names

samples = []
for row in food:
    samples.append(row)
    if len(samples) >= 6:
        break
print("Example categories in this dataset:", class_names[:8], "...")

### Let a pre-trained model label them

<div align="center"><img src="https://raw.githubusercontent.com/ismayc/oreilly-deep-learning-made-simple/main/assets/diagrams/diagram-7944e9c21443.png" alt="Mermaid diagram" width="250"></div>

In [ ]:
from keras.applications.mobilenet_v2 import (
    MobileNetV2, preprocess_input, decode_predictions)

pretrained = MobileNetV2(weights="imagenet")   # trained on millions of images

def predict_label(pil_img):
    img = pil_img.convert("RGB").resize((224, 224))
    batch = np.array(img, dtype="float32")[None]   # [None] wraps one image in a batch of 1
    arr = preprocess_input(batch)                  # the preprocessing this model expects
    top = decode_predictions(pretrained.predict(arr, verbose=0), top=1)[0][0]
    return top[1], round(float(top[2]), 2)   # (label, confidence)

for row in samples[:4]:
    guess, conf = predict_label(row["image"])
    print(f"true: {class_names[row['label']]:<18} model guess: {guess} ({conf})")

In [ ]:
# Show the images with the model's guess -- seeing the picture makes the result land
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 4, figsize=(12, 3.2))
for ax, row in zip(axes, samples[:4]):
    guess, conf = predict_label(row["image"])
    ax.imshow(row["image"]); ax.axis("off")
    ax.set_title(f"true: {class_names[row['label']]}\nguess: {guess} ({conf})", fontsize=9)
plt.tight_layout(); plt.show()

> **Common Pitfall:** Every pre-trained model expects its **own** preprocessing and
> input size (here 224x224 and `preprocess_input`). Skip it and predictions look
> random. When you swap models, swap the matching `preprocess_input`.

> **GenAI tip:** When the pre-trained model guesses a *near-miss* label (say "plate" for
> a pizza), paste its top guesses into an AI assistant and ask why those categories are
> easy to confuse, a fast way to build intuition for what the model actually "sees."

## Exercise 2.1: Fine-Tune a Classifier for *Your* Categories

The pre-trained model knows ImageNet labels, not *your* product categories. Transfer
learning fixes that: keep the model's general "vision," retrain only a small new head
on your few categories.

<div align="center"><img src="https://raw.githubusercontent.com/ismayc/oreilly-deep-learning-made-simple/main/assets/diagrams/diagram-86da90b4e38f.png" alt="Mermaid diagram" width="898"></div>

In [ ]:
# YOUR CALL: pick 3 categories that exist in class_names (print class_names to browse).
# Swap these for whatever "product catalog" you want to recognize.

# Map each Food-101 category index to a fresh 0/1/2 label for our 3-class problem


# Build a small labeled set (~120 images each) -- run this to assemble your data.

In [ ]:
# The transfer-learning move (we'll build this together): keep the base FROZEN so its
# general vision is preserved, and train only the small new head on top.

In [ ]:
# Visualize: 6 validation photos with the model's call (green = right, red = wrong)
val_pred = clf.predict(X_img_val, verbose=0).argmax(axis=1)
fig, axes = plt.subplots(1, 6, figsize=(14, 2.8))
for ax, img, true_i, pred_i in zip(axes, X_img_val[:6], y_img_val[:6], val_pred[:6]):
    ax.imshow(img.astype("uint8")); ax.axis("off")
    ax.set_title(wanted[pred_i], color="green" if true_i == pred_i else "red", fontsize=10)
plt.suptitle("Fine-tuned model on YOUR categories"); plt.tight_layout(); plt.show()

> **Common Pitfall:** Keep the pre-trained base **frozen** (`base.trainable = False`)
> when you have little data. Unfreezing everything with a few hundred images usually
> overfits fast and *erases* the valuable general features.

## Walkthrough + Exercise 2.2: Use GenAI to Explain Layers and Suggest Improvements

Use your AI assistant to demystify a model's architecture and weigh changes for your own
use case. Then test, don't just trust.

**A prompt to start from:**

> Here is my Keras model summary: [paste the output of `clf.summary()`]. Explain what each
> layer does in business terms. Then suggest one change to improve accuracy and one to make
> the model smaller or faster, and describe the trade-offs of each.

In [ ]:
# Scratch space for 2.2 -- paste clf.summary() output here, or try a suggested tweak.

> **GenAI tip:** Apply only ONE suggested change at a time and re-evaluate, so you can see
> what actually helped.

### Interpretation Questions

1. The pre-trained model often guesses a *close* ImageNet label (e.g. "plate",
   "pizza") even before fine-tuning. Why is that "general vision" so reusable?
2. With only 120 images per class, fine-tuning still does well. What does that imply
   for a small business that can't label millions of photos?
3. Where would automated image tagging save real time in your organization?

### Self-Check

By the end of this module, you should be able to:

- [ ] Use a pre-trained CNN to classify images with the correct preprocessing
- [ ] Explain transfer learning in business terms ("reuse, don't reinvent")
- [ ] Fine-tune a frozen base on a small set of your own categories
- [ ] Recognize when freezing the base prevents overfitting

# Module 3: Deep Learning for Text and Customer Sentiment

Direct application: **reading customer sentiment at scale** from reviews and support
text. We use real **Yelp business reviews**.

## Walkthrough 3.1: Sentiment from Reviews with Word Embeddings

### Load real reviews

In [ ]:
from datasets import load_dataset

stream = load_dataset("fancyzhx/yelp_polarity", split="train", streaming=True)
texts, sentiments = [], []
for i, row in enumerate(stream):
    texts.append(row["text"]); sentiments.append(row["label"])  # 0 = negative, 1 = positive
    if i >= 5999:
        break
sentiments = np.array(sentiments, dtype="float32")
print("Loaded", len(texts), "reviews. Example:")
print(texts[0][:160], "...")

### Turn words into numbers (a simple tokenizer)

<div align="center"><img src="https://raw.githubusercontent.com/ismayc/oreilly-deep-learning-made-simple/main/assets/diagrams/diagram-22eff69107d7.png" alt="Mermaid diagram" width="271"></div>

In [ ]:
from collections import Counter

def build_vocab(texts, max_tokens=10000):
    # Count how often each word appears across every review
    counts = Counter(word for text in texts for word in text.lower().split())
    # Give each common word an ID, starting at 2 so 0 stays "padding" and 1 stays "unknown"
    return {word: i + 2 for i, (word, _) in enumerate(counts.most_common(max_tokens - 2))}

def encode(texts, vocab, max_len=150):
    out = np.zeros((len(texts), max_len), dtype="int64")   # rows of zeros = pre-padded
    for i, text in enumerate(texts):
        ids = [vocab.get(word, 1) for word in text.lower().split()][:max_len]  # 1 = unknown word
        out[i, :len(ids)] = ids
    return out

VOCAB_SIZE, MAX_LEN = 10000, 150
vocab = build_vocab(texts, VOCAB_SIZE)
X_text = encode(texts, vocab, MAX_LEN)
X_text_train, X_text_val, y_text_train, y_text_val = train_test_split(
    X_text, sentiments, test_size=0.2, random_state=2026, stratify=sentiments)
print("Encoded shape:", X_text.shape)

> **The TensorFlow-powered alternative:** Keras 3 ships a convenience layer,
> `keras.layers.TextVectorization`, that does this tokenizing for you. It is powered by
> **TensorFlow** under the hood, so it needs TensorFlow available even on the PyTorch
> backend, Colab already ships it. We hand-rolled the tokenizer above to show exactly
> what "turning text into numbers" means; here is the one-layer TensorFlow version doing
> the same job:

In [ ]:
# TensorFlow-backed text vectorization. Colab ships TensorFlow, so this runs as-is.
from keras.layers import TextVectorization

vectorizer = TextVectorization(max_tokens=VOCAB_SIZE, output_sequence_length=MAX_LEN)
vectorizer.adapt(texts)                    # learns the vocabulary from the reviews (via TF)
demo = vectorizer(["the staff were friendly and the food arrived hot"])
print("TextVectorization output shape:", tuple(demo.shape))   # (1, MAX_LEN)
print("First 12 token IDs:", [int(t) for t in demo[0][:12]])

> The hand-built `encode()` and this TensorFlow layer do the same job, map words to
> integer IDs and pad to a fixed length. So the embedding model below trains on either.

### Build an embedding model

### Try it on your own sentences

In [ ]:
def sentiment_of(sentence):
    ids = encode([sentence], vocab, MAX_LEN)
    p = float(text_model.predict(ids, verbose=0)[0][0])
    return f"{'positive' if p > 0.5 else 'negative'} ({p:.2f})"

for s in ["The staff were friendly and the food arrived hot",
          "Waited an hour and the order was completely wrong"]:
    print(sentiment_of(s), "<-", s)

In [ ]:
# A bar chart turns raw probabilities into something you can show a stakeholder
examples = ["The staff were friendly and the food arrived hot",
            "Best brunch in the neighborhood, will be back",
            "Waited an hour and the order was completely wrong",
            "Overpriced and the room was not clean"]
scores = [float(text_model.predict(encode([s], vocab, MAX_LEN), verbose=0)[0][0])
          for s in examples]
colors = ["seagreen" if p > 0.5 else "indianred" for p in scores]
plt.barh(range(len(examples)), scores, color=colors)
plt.yticks(range(len(examples)), [s[:32] + "..." for s in examples], fontsize=8)
plt.axvline(0.5, color="k", ls="--", lw=1)
plt.xlabel("predicted positivity"); plt.xlim(0, 1)
plt.title("Sentiment score per review"); plt.tight_layout(); plt.show()

## Exercise 3.1: Improve and Stress-Test the Model

<div align="center"><img src="https://raw.githubusercontent.com/ismayc/oreilly-deep-learning-made-simple/main/assets/diagrams/diagram-1c818d2e1edd.png" alt="Mermaid diagram" width="365"></div>

In [ ]:
# Lever to pull: a BIGGER vocabulary and a LONGER window. Run it, then compare the
# validation accuracy to the walkthrough -- more capacity doesn't always win.
BIG_VOCAB_SIZE, BIG_MAX_LEN = 15000, 200          # <- try other sizes and see
big_vocab = build_vocab(texts, BIG_VOCAB_SIZE)
X_big = encode(texts, big_vocab, BIG_MAX_LEN)
X_big_train, X_big_val, y_big_train, y_big_val = train_test_split(
    X_big, sentiments, test_size=0.2, random_state=2026, stratify=sentiments)
big_model = keras.Sequential([
    keras.layers.Input(shape=(BIG_MAX_LEN,)),
    keras.layers.Embedding(BIG_VOCAB_SIZE, 32),
    keras.layers.GlobalAveragePooling1D(),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dense(1, activation="sigmoid"),
])
big_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
big_hist = big_model.fit(X_big_train, y_big_train,
                         validation_data=(X_big_val, y_big_val),
                         epochs=5, batch_size=64, verbose=0)
print("Bigger-vocab validation accuracy:", round(float(big_hist.history["val_accuracy"][-1]), 3))

In [ ]:
# YOUR TURN: write tricky reviews (sarcasm, "not great", "would not recommend") and
# see where a word-averaging model slips. Add your own lines to this list.

    # add 2-3 of your own here...

> **GenAI tip:** Paste 3-4 reviews the model gets *wrong* into an AI assistant and ask
> "why might a simple averaging model miss the sentiment here?", great way to surface
> sarcasm, negation, and context limits to discuss with the room.

## Walkthrough + Exercise 3.2: Use GenAI to Summarize Sentiment for Stakeholders

The value of a sentiment model is the decision it informs. Use your AI assistant to turn
raw numbers into a short narrative two different audiences can act on.

**A prompt to start from:**

> Here are my sentiment results: [paste the validation accuracy and 3-4 example reviews
> with their predicted scores]. Write a 3-sentence summary for a non-technical manager,
> what we found, why it matters, and one caution. Then write a more technical version for a
> data analyst.

In [ ]:
# Scratch space for 3.2 -- assemble the numbers/examples you'll paste into your assistant.

> **GenAI tip:** Fact-check every number the assistant repeats before it goes in a deck.
> It will sometimes "round" or invent figures.

### Interpretation Questions

1. Our model **averages** word vectors. What kinds of reviews will that miss
   (think "not great", sarcasm, "would not recommend")?
2. Where would automated sentiment change a decision your team makes today?
3. When is a pre-trained LLM (just prompt it) the better tool than training this model?

### Self-Check

By the end of this module, you should be able to:

- [ ] Turn raw text into padded integer sequences a model can use
- [ ] Build and train an embedding-based sentiment model in Keras
- [ ] Score new sentences and spot where a simple model struggles
- [ ] Read the same model written as a PyTorch `nn.Module` (see the solutions file)

# Module 4: Interpreting and Communicating Deep Learning Results

A model is only as valuable as the trust and clarity you can put around it. We
evaluate the **cancellation model** from Module 1 the way a business actually should.

> **Heads up:** this module reuses the trained `model` and the `X_val` / `y_val` split
> from Module 1, so run those cells first (in the same session) before this one.

## Walkthrough 4.1: Confusion Matrix and Business Metrics

<div align="center"><img src="https://raw.githubusercontent.com/ismayc/oreilly-deep-learning-made-simple/main/assets/diagrams/diagram-7cf2c0216dbc.png" alt="Mermaid diagram" width="502"></div>

In [ ]:
# The bar to beat: always guessing the majority class ("not canceled")

> **Common Pitfall:** Accuracy alone hides the costly errors. Read the confusion
> matrix by **business cost**: a missed cancellation (false negative) is a room you
> didn't resell; a false alarm (false positive) might mean wrongly hassling a guest.

## Exercise 4.1: Tune the Threshold to the Business Trade-off

The model outputs a *probability*. **You** decide where to draw the line, and that
choice is a business decision, not a default of 0.5.

<div align="center"><img src="https://raw.githubusercontent.com/ismayc/oreilly-deep-learning-made-simple/main/assets/diagrams/diagram-df5ac21514b5.png" alt="Mermaid diagram" width="667"></div>

In [ ]:
# The probabilities come from the Module 1 cancellation model.

In [ ]:
# Visualize the trade-off across a fine grid of thresholds.
grid = np.linspace(0.1, 0.9, 33)
prec = [precision_score(y_val, (probs > t).astype(int), zero_division=0) for t in grid]
rec = [recall_score(y_val, (probs > t).astype(int)) for t in grid]
plt.plot(grid, prec, label="precision")
plt.plot(grid, rec, label="recall")
plt.axvline(0.35, color="k", ls="--", lw=1, label="a recall-leaning choice")
plt.xlabel("decision threshold"); plt.ylabel("score"); plt.legend()
plt.title("Precision vs recall is a dial you set to the business cost")
plt.show()

In [ ]:
# YOUR DECISION: missing a cancellation costs ~4x a false alarm, so lean toward recall.
# Change `chosen` and read the consequence out loud in business terms.

> **GenAI tip:** Feed these numbers to an AI assistant and ask for a 3-sentence
> executive summary: the decision, the impact, and the main risk. Then fact-check
> every number it repeats before it goes in a deck.

## Walkthrough + Exercise 4.2: From Feature Importance to an AI-Drafted Executive Summary

First, open the black box. "It's a black box" is the objection you will hear most. The
simplest, model-agnostic way to see what drove the model is **permutation importance**:
shuffle one feature at a time and watch how much accuracy drops. A big drop means the model
leaned heavily on that feature; little drop means it barely used it.

In [ ]:
# Permutation importance: shuffle each feature in turn and measure the accuracy it costs.
# Bigger drop = the model relied on that feature more. (Reuses the Module 1 model + X_val.)

Now you can answer "why did the model decide that?" in plain language: "it flags a booking
mainly on the lead time and the daily rate." For images and text the same idea shows up as
attention maps, over the pixels or words the model leaned on. The goal is never to expose the
math; it is to turn the model's internals into reasons a human can trust.

**Now you try:** turn the metrics *and* that "why" into a decision-ready brief with your AI
assistant. The assistant drafts; you verify and own the recommendation. This is the
make-or-break communication skill of the whole course.

**A prompt to start from:**

> Given these results, precision [X] and recall [Y] at a decision threshold of [T], where
> missing a cancellation costs about 4x a false alarm, plus the top drivers [paste the
> highest-importance features above]. Write a 3-sentence executive summary: the decision, the
> business impact, and the main risk, and end with a one-line plain-English "why."

In [ ]:
# Scratch space for 4.2 -- paste your threshold's precision/recall and the top features here.

> **GenAI tip:** Check the draft against what the model actually showed: the numbers, and
> whether the stated "why" matches the importance chart. Fact-check every figure before it
> goes in a deck.

## Walkthrough 4.3: Package the Model for Deployment (TensorFlow SavedModel)

A trained model only creates value once it's **deployed**. The industry-standard format
for serving is the **TensorFlow SavedModel**, and Keras 3 can export one straight from
the cancellation model we trained on the PyTorch backend (the export goes through
TensorFlow, which Colab already ships):

<div align="center"><img src="https://raw.githubusercontent.com/ismayc/oreilly-deep-learning-made-simple/main/assets/diagrams/diagram-9ecf2030b69e.png" alt="Mermaid diagram" width="941"></div>

In [ ]:
# Export the trained Module 1 cancellation model as a TensorFlow SavedModel.
# `model.export()` writes a TF SavedModel regardless of the active Keras engine.
model.export("hotel_cancel_savedmodel")     # -> a folder a serving stack can load

# Reload it the way a deployment/serving environment would, then predict.
import tensorflow as tf
reloaded = tf.saved_model.load("hotel_cancel_savedmodel")
serving = reloaded.signatures["serving_default"]
print("Served predictions:", serving(tf.constant(X_val[:3], dtype=tf.float32)))

> **Why a SavedModel:** it bundles the network *and* its trained weights into a
> language-agnostic package that TensorFlow Serving, Vertex AI, SageMaker, and most MLOps
> tooling load directly. No Python training code required. Same model you built in
> Module 1; this is how you hand it to production.

### Interpretation Questions

1. As you lower the threshold, recall rises but precision falls. Which way should a
   revenue team lean for cancellations, and why?
2. Write one sentence a non-technical manager would understand describing what this
   model does and what it's worth.
3. What would you monitor after deployment to catch the model going stale (drift)?

### Self-Check

By the end of this module, you should be able to:

- [ ] Build and read a confusion matrix in business terms
- [ ] Compute precision, recall, and F1, and say when accuracy misleads
- [ ] Tune a decision threshold to a stated business cost
- [ ] Turn model results into a clear, honest stakeholder summary
- [ ] Export a trained model as a TensorFlow SavedModel for deployment